In [0]:
from pyspark.sql import functions as F

In [0]:
# Une sale_transactions + sale_items + products em um wide table
# para facilitar a análise de consumo por produto/evento

bronze_sale_tx    = spark.table("workspace.bronze.sales_transactions")
bronze_sale_items = spark.table("workspace.bronze.sales_items")
bronze_products   = spark.table("workspace.bronze.products")
bronze_pos        = spark.table("workspace.bronze.points_of_sale")

silver_sales = (
    bronze_sale_items.alias("si")
    .join(bronze_sale_tx.alias("st"),
          F.col("si.sale_transaction_id") == F.col("st.id"))
    .join(bronze_pos.alias("pos"),
          F.col("st.pos_id") == F.col("pos.id"))
    .join(bronze_products.alias("p"),
          F.col("si.product_id") == F.col("p.id"))
    .select(
        F.col("si.id").alias("sale_item_id"),
        F.col("st.id").alias("transaction_id"),
        F.col("pos.event_id"),
        F.col("pos.name").alias("pos_name"),
        F.col("pos.type").alias("pos_type"),
        F.col("p.id").alias("product_id"),
        F.col("p.name").alias("product_name"),
        F.col("p.category").alias("category"),
        F.col("si.quantity"),
        F.col("si.unit_price"),
        (F.col("si.quantity") * F.col("si.unit_price")).alias("line_total"),
        F.col("st.created_at").alias("sold_at"),
        F.col("st.operator_id"),
    )
)

(silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.sales"))